In [2]:
!pip install catboost

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor, Pool

# 1) Load data
df = pd.read_csv("/content/car_prices.csv", on_bad_lines="skip")

# 2) Basic numeric cleaning
for col in ["sellingprice", "mmr", "odometer", "condition", "year"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["sellingprice", "mmr", "year"])
df = df[(df["sellingprice"] > 0) & (df["mmr"] > 0)].copy()

# 3) Parse datetime (robust)
df["saledate_dt"] = pd.to_datetime(df["saledate"], errors="coerce", utc=True)
df = df.dropna(subset=["saledate_dt"]).copy()
df["saledate_dt"] = df["saledate_dt"].dt.tz_localize(None)

print("saledate_dt dtype:", df["saledate_dt"].dtype)
print("Rows after valid saledate parsing:", len(df))
print("Date range:", df["saledate_dt"].min(), "to", df["saledate_dt"].max())

# 4) Targets: spread
df["dollar_spread"] = df["sellingprice"] - df["mmr"]
df["percent_spread"] = df["dollar_spread"] / df["mmr"]

# 5) Trim outliers
d_lo, d_hi = df["dollar_spread"].quantile([0.01, 0.99])
p_lo, p_hi = df["percent_spread"].quantile([0.01, 0.99])

df = df[
    df["dollar_spread"].between(d_lo, d_hi)
    & df["percent_spread"].between(p_lo, p_hi)
].copy()

print("Rows after trimming:", len(df))

# 6) Feature engineering
df["sale_month"] = df["saledate_dt"].dt.month
df["sale_dow"] = df["saledate_dt"].dt.dayofweek
df["sale_year"] = df["saledate_dt"].dt.year

df["vehicle_age"] = df["sale_year"] - df["year"]
df = df[(df["vehicle_age"] >= 0) & (df["vehicle_age"] <= 30)].copy()

df["mileage_bucket"] = pd.cut(
    df["odometer"],
    bins=[0, 30000, 60000, 100000, 150000, 1000000],
    labels=["0-30k", "30-60k", "60-100k", "100-150k", "150k+"]
)

df["condition_bucket"] = pd.cut(
    df["condition"],
    bins=[0, 2.5, 3.5, 4.5, 6],
    labels=["Poor", "Fair", "Good", "Excellent"]
)

region_map = {
    "TX": "South", "FL": "South", "GA": "South", "NC": "South", "SC": "South",
    "AL": "South", "TN": "South", "LA": "South", "MS": "South", "AR": "South",
    "NY": "Northeast", "NJ": "Northeast", "PA": "Northeast", "MA": "Northeast",
    "CT": "Northeast", "RI": "Northeast", "NH": "Northeast", "VT": "Northeast",
    "ME": "Northeast",
    "IL": "Midwest", "OH": "Midwest", "MI": "Midwest", "IN": "Midwest",
    "WI": "Midwest", "MN": "Midwest", "MO": "Midwest", "IA": "Midwest",
    "KS": "Midwest",
    "CA": "West", "WA": "West", "OR": "West", "NV": "West", "AZ": "West",
    "CO": "West", "UT": "West", "NM": "West", "ID": "West", "MT": "West"
}

df["region_group"] = df["state"].map(region_map).fillna("Other")

# 7) Feature list
FEATURES = [
    "year", "vehicle_age", "make", "model", "trim", "body", "transmission",
    "state", "region_group", "odometer", "mileage_bucket", "condition",
    "condition_bucket", "color", "interior", "sale_month", "sale_dow"
]

FEATURES = [c for c in FEATURES if c in df.columns]

X = df[FEATURES].copy()
y_d = df["dollar_spread"].copy()
y_p = df["percent_spread"].copy()

# 8) Categorical handling
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
cat_idx = [X.columns.get_loc(c) for c in cat_cols]

for c in cat_cols:
    X[c] = X[c].astype("string").fillna("Unknown")

num_cols = [c for c in X.columns if c not in cat_cols]
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors="coerce")
    X[c] = X[c].fillna(X[c].median())

print("Feature count:", len(FEATURES))
print("Categorical columns:", cat_cols)

# 9) Time-based split
df_sorted = df.sort_values("saledate_dt")
split = int(len(df_sorted) * 0.8)

train_idx = df_sorted.index[:split]
test_idx = df_sorted.index[split:]

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_d_train, y_d_test = y_d.loc[train_idx], y_d.loc[test_idx]
y_p_train, y_p_test = y_p.loc[train_idx], y_p.loc[test_idx]

mmr_test = df.loc[test_idx, "mmr"]
price_true = df.loc[test_idx, "sellingprice"]

train_pool_d = Pool(X_train, y_d_train, cat_features=cat_idx)
test_pool_d = Pool(X_test, y_d_test, cat_features=cat_idx)

train_pool_p = Pool(X_train, y_p_train, cat_features=cat_idx)
test_pool_p = Pool(X_test, y_p_test, cat_features=cat_idx)

# 10) Dollar spread model
model_d = CatBoostRegressor(
    loss_function="MAE",
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=6,
    random_seed=42,
    verbose=200,
)

model_d.fit(train_pool_d, eval_set=test_pool_d, use_best_model=True)

pred_d = model_d.predict(X_test)
spread_mae_d = mean_absolute_error(y_d_test, pred_d)

price_pred_d = mmr_test + pred_d
price_mae_d = mean_absolute_error(price_true, price_pred_d)

print("\n=== Dollar Spread Model ===")
print("Spread MAE ($):", round(spread_mae_d, 2))
print("Reconstructed Price MAE ($):", round(price_mae_d, 2))

# 11) Percent spread model
model_p = CatBoostRegressor(
    loss_function="MAE",
    iterations=800,
    learning_rate=0.08,
    depth=6,
    l2_leaf_reg=6,
    random_seed=42,
    verbose=100,
)

model_p.fit(train_pool_p, eval_set=test_pool_p, use_best_model=True)

pred_p = model_p.predict(X_test)
spread_mae_p = mean_absolute_error(y_p_test, pred_p)

price_pred_p = mmr_test * (1.0 + pred_p)
price_mae_p = mean_absolute_error(price_true, price_pred_p)

print("\n=== Percent Spread Model ===")
print("Spread MAE (%):", round(spread_mae_p, 4))
print("Reconstructed Price MAE ($):", round(price_mae_p, 2))

# 12) Save models
model_d.save_model("catboost_dollar_spread.cbm")
model_p.save_model("catboost_percent_spread.cbm")

print("\nSaved: catboost_dollar_spread.cbm, catboost_percent_spread.cbm")

/tmp/ipykernel_32268/2766551208.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["saledate_dt"] = pd.to_datetime(df["saledate"], errors="coerce", utc=True)


saledate_dt dtype: datetime64[ns]
Rows after valid saledate parsing: 558811
Date range: 2014-01-01 01:15:00 to 2015-07-20 19:30:00
Rows after trimming: 538264
Feature count: 17
Categorical columns: ['make', 'model', 'trim', 'body', 'transmission', 'state', 'region_group', 'mileage_bucket', 'condition_bucket', 'color', 'interior']
Train rows: 430470
Test rows: 107618
0:	learn: 938.8023480	test: 943.3206446	best: 943.3206446 (0)	total: 1.03s	remaining: 34m 29s
200:	learn: 793.7894515	test: 827.5234715	best: 827.5234715 (200)	total: 3m 39s	remaining: 32m 42s
400:	learn: 780.2884073	test: 817.8362999	best: 817.8362999 (400)	total: 7m 18s	remaining: 29m 9s
600:	learn: 772.2313744	test: 813.2223174	best: 813.2223174 (600)	total: 10m 56s	remaining: 25m 28s
800:	learn: 766.7924949	test: 810.9054120	best: 810.9054120 (800)	total: 14m 33s	remaining: 21m 48s
1000:	learn: 762.6025227	test: 809.5589203	best: 809.5589203 (1000)	total: 18m 10s	remaining: 18m 8s
1200:	learn: 759.6337530	test: 808.8157

In [ ]:
from google.colab import files
files.download("catboost_dollar_spread.cbm")